In [5]:
import numpy as np
import pandas as pd

## load cmod dataset table from csv using 'pandas'
db = pd.read_csv('data/raw_data/cmod.csv')

## add a "disruptive" column for classification task

# choose a 'tau_class', prior to which states in disruptive shots are considered stable
tau_class = 200e-3  # [s]

# the 'disruptive' condition is defined as finite "time_until_disrupt" < 'tau_class'
disruptive_cond = ( (~np.isnan(db["time_until_disrupt"])) & (db["time_until_disrupt"] < tau_class) )
db["disruptive"] = np.where(disruptive_cond, 1, 0)

## removing shots with cut-off data
## (remove shot if disruptive but does not have data for "time_until_disrupt" < 'tau_class')

# establish list of disruptive shots (ignoring 'disruptive', which will have bad definition for these shots)
disruptive_shot_list = np.unique(db[~np.isnan(db["time_until_disrupt"])]["shot"])

# loop through each disruptive shot to see if data is missing
bad_shots = []
for shot in disruptive_shot_list:
    shot_data = db[db["shot"] == shot]
    if min(shot_data["time_until_disrupt"]) > tau_class:
        bad_shots += [shot]

# remove bad shots from the database
db = db[~db["shot"].isin(bad_shots)]

## remove all rows/points where all relevant features are not populated
## (excluding "time_until_disrupt")

# define relevant features
relevant_features = [
    "ip",
    "ip_error",
    "beta_n",
    "greenwald_fraction",
    "q95",
    "radiated_fraction",
    "z_error",
]

# keep only the rows where there are no nans in the relevant features
db = db[~db[relevant_features].isna().any(axis=1)]

## isolate the flattop phase of the plasma pulse

# specify critical pre-programmed ip ramp-rate, below which defines flattop
crit_dip = 1e3  # [A/s]

# restrict the database to flattop
db_flattop = db[np.abs(db["dipprog_dt"]) <= crit_dip]

## save cleaned version of database
# db_flattop.to_csv("cmod_clean.csv")

print(db_flattop)

              shot  time   a_minor    beta_n    beta_p        bt      chisq  \
20      1120607010  0.46  0.219345  0.286648  0.171107 -5.457756   9.885313   
21      1120607010  0.48  0.219529  0.288905  0.172508 -5.457268   9.618771   
22      1120607010  0.50  0.218793  0.278275  0.165108 -5.455805   9.185632   
23      1120607010  0.52  0.219257  0.282453  0.167467 -5.455804  10.538207   
24      1120607010  0.54  0.218755  0.266011  0.157390 -5.460685  10.059975   
...            ...   ...       ...       ...       ...       ...        ...   
940515  1140211006  1.40  0.224153  0.339973  0.154231  5.386440  16.781702   
940516  1140211006  1.42  0.223878  0.299285  0.135459  5.383023  16.955473   
940517  1140211006  1.44  0.223053  0.303324  0.137733  5.381559  17.233974   
940518  1140211006  1.46  0.222101  0.311538  0.142431  5.382535  17.893890   
940519  1140211006  1.48  0.221867  0.296491  0.135554  5.384487  17.898066   

        dbetap_dt        dip_dt  dip_smoothed  ... 